# Python in the Smoke 💨 🐍 💨
## Applied notebook: natural language processing with spaCy, and London texts

---

**Repo:** [`python-in-the-smoke`](https://github.com/MarkGotham/python-in-the-smoke)

**By:** Mark Gotham

**Licence:** MIT

**Colour and symbol key:** [Itemised and explained here](https://markgotham.github.io/python-in-the-smoke/#note-to-students)

**Assumed knowledge:**

This notebook builds on the foundational Part I notebooks (a to g).
No other applied notebook is required.
Each idea in the table below has appeared before, and now returns in a new setting.

| Part I | Assumed knowledge | How it appears here |
|---|---|---|
| a | text (strings), f-strings | Sentences and texts stored in strings; formatted tables of output |
| b | lists, dictionaries | Lists of entities; a `Counter` is a dictionary that counts |
| c | Booleans, conditionals | Filtering tokens with `if ... and not ...` |
| d | loops | Looping over sentences, tokens and entities |
| e | functions | `model_pairs`, `lemma_counts`, `score` |
| f | packages, dataframes | `pandas` tables of entities: selection and category counts |
| g | files and paths | Not needed: all texts are in the notebook, or downloaded in Part 5 |

**New here, and explained where it appears: sets** (Part 2).

Retrieving data from the web (APIs and scraping) is taught elsewhere: Part 5 only *applies* it.

---
## 🟢 Information: Why this topic, and why this design?

**The cultural question.**
How does writing about London change across about 260 years?
And what does a language model trained on *modern text* make of each period?
The second question is the real subject: it teaches you to treat a model's output as a claim to test, not a fact.

**The plan.**
1. **Part 1:** simple modern sentences, where the model does well.
2. **Part 2:** a way to measure "well", by comparing the model with your own judgement.
3. **Part 3:** three short texts from three periods: does the model fail only on old text?
4. **Part 4:** rules that patch some of the failures.
5. **Part 5:** a blueprint for downloading real texts, to try it all on.

📝 **Parts 1 to 4 use invented texts.**
They were written for this course, in the manner of each period.
The people in them do not exist (the London places do).
That keeps this repository free of copyright questions, and gives us texts with known features.
Real texts come in Part 5, where *you* download them.

---
## 🟢 Information: What is an NLP pipeline?

Natural language processing (NLP) turns text into structured data.
A spaCy **pipeline** is a sequence of steps applied to a text:

1. **Tokenise:** split the text into tokens (roughly words and punctuation).
2. **Tag:** give each token a part of speech (noun, verb, ...) and a **lemma** (dictionary form: "walked" → "walk").
3. **Parse:** work out sentence boundaries and grammatical links.
4. **Recognise entities:** find names of people, places, organisations and so on.

Each step is a *statistical model* trained on example text.
spaCy's small English model is trained on modern written English
(blogs, news, comments; the [model page](https://spacy.io/models/en) lists the training sources).
So we expect it to do well on modern English like that;
we cannot assume it will work as well with a seventeenth-century diary.
But we're scientists right?
So actually we assume almost nothing and will test all expectations.

---
## 🟢 Information: Set up

We need `spacy` and its small English model.
- `spacy` is the library.
- `en_core_web_sm` is the model: the trained data. It is a separate download.

Run the cells below once. On Colab, spaCy is often already installed, but the model may not be.

📝 We ask for spaCy version 3.8, because the outputs quoted in the solutions come from that version and its model.
Another version may give different results (that is itself a lesson).
If the install fails on a very new Python, install the latest spaCy instead and expect differences.

In [ ]:
%pip install "spacy>=3.8,<3.9"

In [ ]:
import spacy
import spacy.cli

spacy.cli.download("en_core_web_sm")

📝 If `spacy.load` below says it cannot find the model after you install it,
restart the kernel (on Colab: Runtime → Restart)
and run the notebook again from the imports.

In [ ]:
from collections import Counter

import pandas as pd
import spacy

In [ ]:
nlp = spacy.load("en_core_web_sm")
print("spaCy", spacy.__version__, "| model", nlp.meta["name"], nlp.meta["version"])

📝 **Record your versions.** Results depend on the spaCy and model versions, so write them down beside any findings.
They are printed above.

---

# Part 1: A model at its best

## 🟢 Information: Run the pipeline on simple sentences

Here are four invented sentences about modern London life, in the kind of text the model was trained on.

`nlp(text)` runs every step and returns a `Doc`, which holds all the results.
- Iterating over a `Doc` gives **tokens**.
- `doc.sents` gives **sentences**.
- `doc.ents` gives **named entities**.

In [ ]:
simple_sentences = [
    "Priya Shah opened the new gallery in Peckham on Tuesday.",
    "The British Museum in London welcomed visitors from Japan.",
    "Daniel Okoro lives in Brixton and works at the University of London.",
    "Amina Hassan visited Paris last summer.",
]

doc = nlp(simple_sentences[0])
print(len(doc), "tokens")

Token attributes we use:
- `token.text`: the original text.
- `token.lemma_`: the dictionary form.
- `token.pos_`: the part of speech.
- `token.is_stop`: whether it is a very common word ("the," "and").
- `token.is_alpha`: whether it is made of letters (not punctuation or numbers).

The trailing underscore matters: `token.lemma_` is text, while `token.lemma` is an internal number.

In [ ]:
for token in doc:
    print(f"{token.text:10} {token.lemma_:10} {token.pos_:6} stop={token.is_stop}")

A **named entity** is a name the model recognises, with a **label** for its type:
- `PERSON`: people.
- `GPE`: countries, cities, states ("geopolitical entities").
- `LOC`: other locations (mountains, rivers).
- `FAC`: buildings, streets, bridges.
- `ORG`: companies and institutions.
- `DATE`, `TIME`: times.

`spacy.explain("GPE")` describes any label.
These categories are the model's, not facts about the text: for example, is a street in London a `FAC`, a `GPE` or a `LOC`?

In [ ]:
for label in ["PERSON", "GPE", "LOC", "FAC", "ORG"]:
    print(label, "=", spacy.explain(label))

In [ ]:
for sentence in simple_sentences:
    doc = nlp(sentence)
    print(sentence)
    print("   entities:", [(ent.text, ent.label_) for ent in doc.ents])

On these sentences the model does what we hoped: people, places and organisations are found and labelled sensibly.
Here, the model is working as intended.
The rest of the notebook asks how far that can be extended and how to handle failures in real data.

---
### 🟣 Core Exercise: Your own sentences

1. Write three sentences about London in your own words, using invented people and real places.
2. *Before* running them, predict in the Markdown cell below the entities and labels you expect.
3. Run the pipeline on each and print its entities.
4. Compare with your predictions. Where did the model differ?

*Your predictions and comparison here.*

---
---
---

### ✅ Solution: Your own sentences

In [ ]:
my_sentences = [
    "Jamal Clarke cycled from Hackney to Greenwich on Sunday.",
    "The Barbican Centre is hosting a concert for Hackney Council.",
    "Dr Lucy Evans moved to Edinburgh in 2019.",
]

for sentence in my_sentences:
    doc = nlp(sentence)
    print(sentence)
    print("   entities:", [(ent.text, ent.label_) for ent in doc.ents])

The answers depend on your sentences (and the model); here is what the model v3.8.0 gave for ours:
- `Jamal Clarke` was split into `Jamal` (labelled `GPE`) and `Clarke` (`ORG`), and `Hackney` was labelled `ORG`.
- `The Barbican Centre` and `Hackney Council` were both found as `ORG`.
- `Dr Lucy Evans` became `Lucy Evans`, a `PERSON`, with the title left out.

So even simple sentences can go wrong, here with a name that is less common, or that could also be a place or an organisation.
Questions of representation are self-evident.

📝 Note:
A prediction is only useful if you commit to it first: it is easy to explain any output after the fact.
If every one of your sentences worked, try "harder tasks" such as:
- lesser known and/or more unusual cases
- known cases of potential confusion e.g., names that are also places words.

---

# Part 2: Measuring "well"

## 🟢 Information: Evaluate against your own judgement

How good are these entities?
To find out, **hand-label** a sample (the "gold standard") and compare.

For each sentence, list the entities *you* think are there, as pairs of `(text, label)`.
Then compare with the model's entities:
- **True positive (tp):** in both lists.
- **False positive (fp):** the model found it, you did not (a spurious entity, or the wrong label).
- **False negative (fn):** you found it, the model did not (a miss, or the wrong label).

There are two common measures of success given this data which are different questions:

| Name          | Function       | Question                                        |
|---------------|----------------|-------------------------------------------------|
| **Precision** | tp ÷ (tp + fp) | Of what the model found, how much was right?    |
| **Recall**    | tp ÷ (tp + fn) | Of what was there, how much did the model find? |

## 🟢 New topic: sets

To compare two lists of pairs we use a new data type, the **set**.
A set is a collection with **no repeats and no order**.
It is written with curly brackets (like a dictionary, but with no colons), or built from a list with `set(...)`.
Sets have operations for comparing collections:
- `a & b` gives the items in both.
- `a - b` gives the items in `a` but not in `b`.

In [ ]:
a = {"Peckham", "Brixton", "Paris"}
b = {"Peckham", "Paris", "Lagos"}

print(a & b)
print(a - b)
print(set(["Paris", "Paris", "Lagos"]))

📝 Sets have no order, so the items may print in a different order on your computer.
Note that the last line has two `"Paris"` and the set keeps one.

## 🟢 Information: The `score` function

We turn the idea into two functions.
- `model_pairs` collects the model's entities as `(text, label)` pairs.
  We keep only people, places and organisations, and we ignore a leading "the", because annotators disagree about it.
  Both are **decisions**, and they affect the score.
- `score` compares the model's pairs with the gold pairs, using sets.

In [ ]:
KEEP = {"PERSON", "ORG", "GPE", "LOC", "FAC"}  # ignore dates, times, numbers


def model_pairs(span):
    pairs = []
    for ent in span.ents:
        if ent.label_ in KEEP:
            name = ent.text
            if name.lower().startswith("the "):
                name = name[4:]
            pairs.append((name, ent.label_))
    return pairs


def score(predicted, gold):
    predicted, gold = set(predicted), set(gold)
    tp = len(predicted & gold)
    fp = len(predicted - gold)
    fn = len(gold - predicted)
    precision = tp / (tp + fp) if (tp + fp) > 0 else None
    recall = tp / (tp + fn) if (tp + fn) > 0 else None
    return {"tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall}

📝 Because we use sets, a pair that occurs twice counts once, and its position in the text is ignored.
That is fine for a few sentences, but it would hide repeated mistakes in a long text (a text that names "London" fifty times would count it once).

Here is a worked example on the four simple sentences.
The gold list is what we think is there; the predicted list is what the model found.

In [ ]:
gold_simple = [
    ("Priya Shah", "PERSON"), ("Peckham", "GPE"),
    ("British Museum", "ORG"), ("London", "GPE"), ("Japan", "GPE"),
    ("Daniel Okoro", "PERSON"), ("Brixton", "GPE"), ("University of London", "ORG"),
    ("Amina Hassan", "PERSON"), ("Paris", "GPE"),
]

predicted_simple = []
for sentence in simple_sentences:
    predicted_simple.extend(model_pairs(nlp(sentence)))

score(predicted_simple, gold_simple)

A perfect score, on ten entities.
That is encouraging but says little: these sentences were chosen because they are easy.
The useful question is where the score stops being perfect.

In [ ]:
hard_sentences = [
    "Maria bought tickets for the Royal Albert Hall in January.",
    "We met Dr Okafor at King's Cross station at 9am.",
    "Mr Evans took the Northern line to Camden.",
]

for sentence in hard_sentences:
    doc = nlp(sentence)
    print(sentence)
    print("   model:", [(ent.text, ent.label_) for ent in doc.ents])

---
### 🟣 Core Exercise: Score the model on harder sentences

Here are three more sentences, printed above with the model's entities.
The model's side of the score is produced by the code; the **gold list is your job**.

1. Read the sentences and the model's entities above.
2. Write your own gold list of `(text, label)` pairs for all three, using spaCy's label names and keeping to people, places and organisations.
3. Collect the model's pairs with `model_pairs`, for all three sentences together.
4. Call `score(predicted, gold)` and report precision and recall.
5. Look at the model's *full* entities (`doc.ents`): which of its errors does the score hide or reveal?

*Your answer here.*

---
---
---

### ✅ Solution: Score the model on harder sentences

In [ ]:
# These are one reader's judgements. Replace them with your own.
gold_hard = [
    ("Maria", "PERSON"), ("Royal Albert Hall", "FAC"),
    ("Okafor", "PERSON"), ("King's Cross", "FAC"),
    ("Evans", "PERSON"), ("Northern line", "FAC"), ("Camden", "GPE"),
]

predicted_hard = []
for sentence in hard_sentences:
    predicted_hard.extend(model_pairs(nlp(sentence)))

score(predicted_hard, gold_hard)

With the gold list above and model 3.8.0 we get 2 true positives, 3 false positives and 5 false negatives: precision 0.40 and recall 0.29.
The model's errors:
- `Maria` is labelled `NORP` (a nationality or group), which `KEEP` drops, so it counts as a miss.
- `Royal Albert Hall` becomes `Albert Hall`, labelled `PERSON`.
- `Dr Okafor` is not found at all.
- `King's Cross` is labelled `ORG`, where we said `FAC`: a wrong label counts as one false positive *and* one false negative.
- `Northern line` is cut to `Northern`, labelled `LOC`.

Our gold list is one reader's decisions. `ORG` for King's Cross could be defended, and another choice would change the score.
Two honest caveats to state with any score: the sample is tiny, and *you* decide what counts as an entity and which label is correct.
Another person could disagree with some of your gold labels, and then the scores would change.

---

# Part 3: Three periods

## 🟢 Information: Our three texts

Each text is **invented**, in the manner of its period.
They are short (under 200 tokens each): too short for reliable statistics, but long enough to see patterns.

| | Written in the manner of | Period |
|---|---|---|
| Period A | a diary | 1660s |
| Period B | a London novel | 1920s |
| Period C | a news report | recent |

Period C is closest to the model's training text, so we can ask whether the problems we find are about *time* or about something else.
The people are invented; the London places are real.

📝 `"""`, three quote marks, lets a string run over several lines and contain quotation marks.

In [ ]:
period_texts = {
    "Period A": """Up betimes, and walked from the Temple through Cheapside with Mr Hale, who was in a great pother about his debts. Thence by water to Whitehall, where we waited most of the morning at the Navy Office and were much vexed. Home to dinner, and my wife very merry, which pleased me exceedingly. After dinner to Drury Lane with Mrs Quill, to see a new play at the King's Company. The house was full, and the play, I think, but a poor thing, though the music was mighty fine and the actresses very handsome. Thence home by coach, being afeard of the dark and the cutpurses about Bow, and so to bed. A sad day today, for news came that poor Tom Marlow is dead of the plague in Southwark, and the whole city is full of fear. I could not eat, and to bed in great trouble of mind.""",
    "Period B": """It was evening, and winter, and Mrs Alder, who had no need of anything at all, put on her coat and went out into Oxford Street. The lamps were lovely, the crowd moved pleasantly, and it was absurd to be thinking of a pair of gloves. She turned down towards Bond Street, where the windows at Harrods and the little jewellers glittered like a row of small lit stages. Waterloo, she thought, and Marylebone, and Bow; the names ran through her like music. A man in a shabby coat, perhaps a clerk from the Temple, stood quite still at the corner of Piccadilly and stared at nothing. Mr Penhale, she supposed, or someone very like him, tired and rather dreadful. Then the rain began, and the gaiety went out of the street. She was cold and a little frightened, and she hurried home.""",
    "Period C": """The Southbank Centre has announced a new season of free concerts, starting on 14 March. Organisers expect more than 20,000 visitors. Amara Bello, the festival's director, told reporters that tickets for the Royal Festival Hall would go on sale on Friday. The Elizabeth line will run late on Saturday nights, and Transport for London has promised extra buses from Waterloo station. Meanwhile, a gallery in Peckham is showing work by young artists from Lagos and Manchester. "People come for the music, but they stay for the food," said Mr Jones, who runs a cafe near Borough Market. Entry costs £12 for adults, and children go free.""",
}

docs = {}
for period, text in period_texts.items():
    docs[period] = nlp(text)
    print(period, "|", len(docs[period]), "tokens |", len(list(docs[period].sents)), "sentences")

## 🟢 Information: Tokens across periods

First, the tokens of Period C: modern text, so we expect few surprises.

In [ ]:
doc_c = docs["Period C"]
for token in doc_c[:15]:
    print(f"{token.text:12} {token.lemma_:12} {token.pos_:6} stop={token.is_stop}")

---
### 🟣 Core Exercise: Tokens in Period A

Print the same table for the first 15 tokens of `"Period A"`.
Then write one observation in the Markdown cell below: did the lemma or part of speech look wrong for any token?

*Your observation here.*

---
---
---

### ✅ Solution: Tokens in Period A

In [ ]:
doc_a = docs["Period A"]
for token in doc_a[:15]:
    print(f"{token.text:12} {token.lemma_:12} {token.pos_:6} stop={token.is_stop}")

Look for older word forms or unusual names that the lemmatiser gets wrong,
and for punctuation or capitalised words tagged with the wrong part of speech.
In Period A, `betimes` is given the lemma `betime` and tagged as a noun (it is an old adverb).
Compare the Period C table, where nothing stands out.

Spotting errors by eye is the first step of evaluating a model, and so is noticing when there are none to find.

---
## 🟢 Information: Word frequencies, fairly

To see what each text talks about, count the lemmas.
We skip stop words and anything that is not made of letters.
`Counter` is a dictionary that counts for us: `counts[word] += 1`.

⚠️ Raw counts are not comparable if the texts differ in length.
Divide by the number of tokens and multiply by N to get a rate **per N tokens**.
By convention, we use $N=1,000$ to get a rate **per 1,000 tokens**.

In [ ]:
def lemma_counts(doc):
    counts = Counter()
    for token in doc:
        if token.is_alpha and not token.is_stop:
            counts[token.lemma_.lower()] += 1
    return counts


for period, doc in docs.items():
    print(period, lemma_counts(doc).most_common(8))

---
### 🟣 Core Exercise: Compare fairly

For each period, print the ten most common lemmas with their count **and** their rate per 1,000 tokens.
Then write two sentences in the Markdown cell below:
1. What does each text seem preoccupied with?
2. Why are the *rates* fairer than the counts?

*Your two sentences here.*

---
---
---

### ✅ Solution: Compare fairly

In [ ]:
for period, doc in docs.items():
    print(period, f"({len(doc)} tokens)")
    for lemma, count in lemma_counts(doc).most_common(10):
        rate = count / len(doc) * 1000
        print(f"  {lemma:12} {count:3}  {rate:5.1f} per 1,000")

Counts grow with text length, so a longer text looks "more preoccupied" with everything.
Rates put the texts on the same scale.
Even so, texts this short give noisy rates: one extra occurrence moves the rate a lot.
The most frequent lemmas are a starting point for reading, not a finding.

---
## 🟢 Information: Entities across periods

We collect every entity from every text into one dataframe.

In [ ]:
rows = []
for period, doc in docs.items():
    for ent in doc.ents:
        rows.append({
            "period": period,
            "entity": ent.text,
            "type": ent.label_,
            "sentence": ent.sent.text.strip(),
        })

entities = pd.DataFrame(rows)
entities.head(10)

In [ ]:
entities.groupby("period")["type"].value_counts()

---
### 🟣 Core Exercise: Predict, then check

*Before* looking at more output, predict in the Markdown cell below:
- which period the model will handle best for **place names**, and which worst
- one London place name you expect it to get wrong, and why

Then display the entities labelled `GPE`, `LOC` or `FAC` for each period, and check your predictions.
Does every place name in the texts appear in your table?

*Your predictions here.*

---
---
---

### ✅ Solution: Predict, then check

In [ ]:
places = entities[entities["type"].isin(["GPE", "LOC", "FAC"])]
places[["period", "entity", "type"]]

A prediction is only useful if you commit to it first.
Look for real London places that never appear in the table, for the same place receiving different labels, and for places labelled as people or organisations.
The table only shows what the model *found*: to find the misses, read the texts.

Several London names are genuinely ambiguous: "Waterloo" and "Temple" are also people and institutions, and "Bow" is also an ordinary word.
With model 3.8.0:
- `Waterloo` comes out as `PERSON` in **both** Period B and Period C, so it is missing from the table.
- `Elizabeth line` loses its second word and is labelled `PERSON` (Period C).
- `Penhale`, a man's name, is labelled `GPE` (Period B).
- `Marylebone`, `Bow` and `Piccadilly` (Period B) and `Whitehall` (Period A) do not appear as places.
- Period C gives only three clean places: `Peckham`, `Lagos` and `Manchester`.

The oldest text is not the only one with problems.

---
## 🟢 Information: Score each period

We now score a sample from each period, using the functions from Part 2.
To keep the sample reproducible and not cherry-picked, we use the first five sentences of at least eight tokens.
Five sentences is far too few for a reliable score, but enough to see how the method works.

In [ ]:
def sample_sentences(doc, how_many=5):
    chosen = []
    for sent in doc.sents:
        if len(sent) >= 8 and len(chosen) < how_many:
            chosen.append(sent)
    return chosen


for period, doc in docs.items():
    print(period)
    for i, sent in enumerate(sample_sentences(doc)):
        print(" ", i, sent.text.strip())
        print("     model:", [(ent.text, ent.label_) for ent in sent.ents])

---
### 🟣 Core Exercise: Score the model on each period

1. Read the sample sentences above.
2. For each period, write your own gold list of `(text, label)` pairs (people, places and organisations).
3. For each period, collect the model's pairs with `model_pairs` across the five sentences.
4. Call `score(predicted, gold)` for each period and report precision and recall.
5. In two sentences: do the scores follow the order of the periods in time?

*Your two sentences here.*

---
---
---

### ✅ Solution: Score the model on each period

In [ ]:
# One reader's judgements. Replace them with your own.
gold = {
    "Period A": [
        ("Temple", "FAC"), ("Cheapside", "FAC"), ("Mr Hale", "PERSON"),
        ("Whitehall", "FAC"), ("Navy Office", "ORG"),
        ("Drury Lane", "FAC"), ("Mrs Quill", "PERSON"), ("King's Company", "ORG"),
    ],
    "Period B": [
        ("Mrs Alder", "PERSON"), ("Oxford Street", "FAC"), ("Bond Street", "FAC"),
        ("Harrods", "ORG"), ("Waterloo", "FAC"), ("Marylebone", "FAC"),
        ("Bow", "FAC"), ("Piccadilly", "FAC"), ("Temple", "FAC"),
    ],
    "Period C": [
        ("Southbank Centre", "ORG"), ("Amara Bello", "PERSON"), ("Royal Festival Hall", "FAC"),
        ("Elizabeth line", "FAC"), ("Transport for London", "ORG"), ("Waterloo station", "FAC"),
        ("Peckham", "GPE"), ("Lagos", "GPE"), ("Manchester", "GPE"),
        ("Jones", "PERSON"), ("Borough Market", "FAC"),
    ],
}

for period, doc in docs.items():
    predicted = []
    for sent in sample_sentences(doc):
        predicted.extend(model_pairs(sent))
    print(period, score(predicted, gold[period]))

With these gold lists and model 3.8.0:

|          | tp | fp | fn | precision | recall |
|----------|----|----|----|-----------|--------|
| Period A | 3  | 2  | 5  | 0.60      | 0.38   |
| Period B | 4  | 3  | 5  | 0.57      | 0.44   |
| Period C | 6  | 4  | 5  | 0.60      | 0.55   |

Broadly we an summarise as follows:
- _Recall_ rises as the texts get more recent (0.38, 0.44, 0.55),
- _Precision_ is about the same (0.57 to 0.60),
- The model gets a lot wrong even in Period C.
- With only five sentences, one entity moves recall by roughly 0.1, so we cannot tell whether the trend is real.
- The "gold" lists are also our own somewhat subjective decisions:
another reader might label `Waterloo station` as `GPE` or `LOC`, and the scores would change.

Compare with Part 2: on the simple sentences the model scored 1.0,
but on these London place names it struggles in *every* period.
So the _kind_ of entity matters at least as much as the _period_.

---
### 🟣 Core Exercise: Where does it go wrong?

Look at the model's entities for **Period C** and **Period A** again (in the sample sentences above, and in the `entities` table).
Do not score them. Instead, sort every error you notice into one of these types, and give one example of each (if the samples have no example of a type, say so):
- **missed** (an entity the model did not find)
- **spurious** (something that is not an entity)
- **wrong label** (found, but typed wrongly)
- **wrong boundary** (the right thing, but the span is too short or too long)

Which types appear in *both* periods?

*Your error types and examples here.*

---
---
---

### ✅ Solution: Where does it go wrong?

With model 3.8.0:

| Type | Period C | Period A |
|---|---|---|
| **Wrong label** | `Amara Bello` as `ORG`; `Waterloo` as `PERSON`; `Royal Festival Hall` as `EVENT` | `Drury Lane` as `PERSON` |
| **Wrong boundary** | `Elizabeth` for `Elizabeth line` | `the Temple through Cheapside` as one `LOC` span |
| **Missed** | nothing clear-cut (the misses show up as wrong labels and boundaries) | `Whitehall`; `Mr Hale` |
| **Spurious** | `a new season` as `DATE` (debatable) | `A sad day` as `DATE` |

Wrong labels and wrong boundaries appear in **both** periods, and both texts contain a debatable `DATE`.
Only Period A has clear misses.
Naming the *kind* of error tells you what to fix, and whether it can be fixed with rules (Part 4) or needs better training data.
Your output may differ with another model version, and short invented texts cannot show every kind of error.

---

# Part 4: Patching with rules

## 🟢 Information: Rules before statistics

A rule-based component can catch names the statistical model misses.
spaCy's `entity_ruler` lets us add patterns: here, a list of London places with our own label.
We place it *before* the `ner` step, so that our rules take priority.

---
### ⭐ Optional Exercise: Add London rules

1. Choose at least five London places from the three texts that the model missed or mislabelled.
2. Add them to a **new copy** of the pipeline using `entity_ruler`, placed *before* the `ner` step.
3. Run it on all three texts and print the entities with your new label.
4. State one thing your rule list assumes, and one reason why testing it on these same texts is not a fair test.

*Your answers here.*

---
---
---

### ✅ Solution: Add London rules

In [ ]:
nlp_ruled = spacy.load("en_core_web_sm")
ruler = nlp_ruled.add_pipe("entity_ruler", before="ner")
ruler.add_patterns([
    {"label": "LONDON_PLACE", "pattern": "Cheapside"},
    {"label": "LONDON_PLACE", "pattern": "Drury Lane"},
    {"label": "LONDON_PLACE", "pattern": "Whitehall"},
    {"label": "LONDON_PLACE", "pattern": "Temple"},
    {"label": "LONDON_PLACE", "pattern": "Waterloo"},
    {"label": "LONDON_PLACE", "pattern": "Marylebone"},
    {"label": "LONDON_PLACE", "pattern": "Piccadilly"},
    {"label": "LONDON_PLACE", "pattern": "Elizabeth line"},
    {"label": "LONDON_PLACE", "pattern": "Royal Festival Hall"},
    {"label": "LONDON_PLACE", "pattern": "Borough Market"},
])

for period, text in period_texts.items():
    doc_ruled = nlp_ruled(text)
    print(period, [ent.text for ent in doc_ruled.ents if ent.label_ == "LONDON_PLACE"])

With these ten rules, every listed place is found as `LONDON_PLACE` where it appears: `Drury Lane` is no longer a `PERSON`, `Whitehall` is no longer missed, `Elizabeth line` keeps its second word, and `Waterloo` is a place in both Period B and Period C.

Rules are precise (they match exactly what you list) but only as good as the list.
The list is also a set of **decisions**: which places count as London, which spellings are included (older texts use other spellings), and whether a name that means something else elsewhere should still match.
A rule for `Bow` would also match a capitalised ordinary word, and `Temple` would match a person called Temple, so rules need testing too.

⚠️ Testing on the same texts we wrote the rules from is circular: of course they work there.
A fair test needs a *new* text, which is a good use of a text from Part 5.

---

# Part 5: Real texts, a blueprint

## 🟢 Information: Downloading a real text

So far every text was invented.
To try the same steps on a real text, you can **download it while the notebook runs**.
Nothing is stored in this repository, and the cells below fall back to invented text if the download fails.

⚠️ **Note:**
- This part is a **blueprint**, not a recipe: websites change, and you will want to adapt it to your own sources.
- Both
[APIs](./apis_json.ipynb) and
[web scraping](./web_scraping.ipynb).
are taught elsewhere.
Click on those links to revisit; here we only apply them.

⚠️ **Before you download anything:**
- **Copyright.** We think the example texts below are out of copyright, but that depends on the work, the *edition* and the country, so **always check for yourself**. Also read the site's own terms: some ask for attribution or limit re-use.
- **Be polite to servers.** Send an identifying `User-Agent`, download once rather than on every run, and do not loop over many pages.
- **Do not commit** downloaded texts to a public repository unless you have checked that you may.
- **Record** where each text came from, which edition, and under what terms, as you would for any source.

In [ ]:
import json
import re
from urllib.parse import urlencode
from urllib.request import Request, urlopen
from bs4 import BeautifulSoup

USER_AGENT = "Student using the python-in-the-smoke-notebook (https://github.com/MarkGotham/python-in-the-smoke)"

def fetch(url):
    request = Request(url, headers={"User-Agent": USER_AGENT})
    with urlopen(request, timeout=30) as response:
        return response.read().decode("utf-8")

## 🟢 Information: Example 1, Wikisource

[Wikisource](https://en.wikisource.org) is a library of transcribed texts with a web API.
Here we ask it for the plain text of a page.
The page title below (the opening section of a 1925 novel) is only an example: adapt it.

In [ ]:
def wikisource_text(title):

    # Fast path: TextExtracts (no BS needed if it works)
    params = {"action": "query", "prop": "extracts", "explaintext": 1,
              "titles": title, "format": "json", "redirects": 1}
    API_URL = "https://en.wikisource.org/w/api.php"
    data = json.loads(fetch(API_URL + "?" + urlencode(params)))
    page = next(iter(data["query"]["pages"].values()))
    if page.get("extract"):
        return page["extract"].strip()

    # Slow path: Parse API + BeautifulSoup
    params = {"action": "parse", "page": title, "prop": "text",
              "format": "json", "redirects": 1, "disableeditsection": 1}
    data = json.loads(fetch(API_URL + "?" + urlencode(params)))
    if "error" in data:
        raise LookupError(f"Page not found: {title!r}")

    soup = BeautifulSoup(data["parse"]["text"]["*"], "html.parser")
    for tag in soup(["style", "script", "nav", "sup"]):
        tag.decompose()
    for tag in soup.find_all(class_=re.compile(r"ws-noexport|noprint")):
        tag.decompose()
    content = soup.find("div", class_="mw-parser-output") or soup.body
    return content.get_text("\n").strip()

In [ ]:
try:
    wikisource = wikisource_text("Mrs. Dalloway/Section 1")
    print(len(wikisource.split()), "words")
    print(wikisource[:300])
except Exception as error:
    print("Could not download from Wikisource:", error)
    wikisource = None

If this returns a number of words (`> 0`) and some human-readable text, then it's worked.

You can retrieve the contents with the variable created in the above cell called `wikisource`.

## 🟢 Information: Example 2, Project Gutenberg

[Project Gutenberg](https://www.gutenberg.org) offers plain-text books at addresses that follow a pattern, using the book's number (from its page: `gutenberg.org/ebooks/<number>`).
Gutenberg files begin and end with licence text, which we must remove before analysis, or it would be analysed as if it were part of the book.
The number below (a volume of Pepys's diary) is only an example.

In [ ]:
def strip_gutenberg(text):
    start = text.find("*** START OF")
    if start != -1:
        text = text[text.find("\n", start) + 1:]
    end = text.find("*** END OF")
    if end != -1:
        text = text[:end]
    return text.strip()


try:
    raw = fetch("https://www.gutenberg.org/cache/epub/4125/pg4125.txt")
    gutenberg = strip_gutenberg(raw)
    print(len(gutenberg.split()), "words")
    print(gutenberg[:300])
except Exception as error:
    print("Could not download from Project Gutenberg:", error)
    gutenberg = None

📝 Look at what you actually downloaded.
The start of a file may be a title page or an editor's preface, not the part you want, and old editions often add the editor's own notes, which are not the original writer's words.

## 🟢 Information: Other sources

The same pattern (fetch, clean, analyse) applies elsewhere.
- **Standard Ebooks** publishes carefully produced editions of public-domain books, as web pages and ebook files.
- **The Internet Archive** holds scans of a huge range of books, with machine-read (OCR) text. OCR mistakes then become part of your data, and copyright status varies from item to item.

Check each source's terms, and expect to adapt the cleaning step: each source has its own layout.

## 🟢 Information: Choose a text and run the pipeline

The cell below picks whichever download worked, or falls back to the invented Period B text, and says so.
It analyses only the first 5,000 characters, to keep it quick.

In [ ]:
if wikisource is not None:
    real_text, source = wikisource, "Wikisource"
elif gutenberg is not None:
    real_text, source = gutenberg, "Project Gutenberg"
else:
    real_text, source = period_texts["Period B"], "the invented Period B (downloads failed)"

print("Using text from", source)
real_doc = nlp(real_text[:5000])
print(len(real_doc), "tokens")
print(Counter(ent.label_ for ent in real_doc.ents).most_common(6))
print([ent.text for ent in real_doc.ents if ent.label_ in ("GPE", "LOC", "FAC")][:10])

---
### ⭐ Optional Exercise: Try it on a real text

Apply Parts 2 to 4 to your real text:
1. Print the first five sentences of at least eight tokens, with the model's entities.
2. Hand-label them and call `score`.
3. List the errors by type.
4. Say what you would need to record about this text before using the results in a research claim.

*Your findings here.*

---
---
---

### ✅ Solution: Try it on a real text

In [ ]:
for sent in sample_sentences(real_doc):
    print(sent.text.strip()[:100])
    print("   model:", [(ent.text, ent.label_) for ent in sent.ents])

Your results depend on the text you downloaded, so there is no single answer.
Typical things to find: headings, page numbers or markup that the model treats as text; place names that are ambiguous or spelt differently; and a score that is lower than on the simple sentences of Part 1.
To record: the work, the edition, the source and its terms, the date downloaded, which part you analysed, and the spaCy and model versions.

---
## 🟢 Information: Where next?

The idea of this notebook was to treat a model's output as a claim to test.
Here are three ways to take that idea further.

- **Sentiment analysis.**
This moves the task away from parts of speech to consider instead how positive or negative it is.
A brief demo follows this cell.
A larger project could combine this with a larger set of texts,
perhaps those that already have human labels (for example, labelled film reviews),
and report how often the tool agrees with the labels.
- **Another language, another model.** spaCy has models for many languages, including Chinese (`zh_core_web_sm`).
Run the same steps and sort the errors in the same way.
Note what changes: the Chinese model has no lemmatiser, so `token.lemma_` is empty, and you would count `token.text` instead.
- **Other kinds of texts such as song lyrics.**
We've considered diaries, novels, and modern web text, but there's much more out there,
such as poetry and song lyrics for example.
This
Note: Full lyrics are almost always in copyright, so they cannot simply be downloaded and shared.
Research datasets often get round this by releasing only word counts (the musiXmatch dataset, for example).
That's enough for word frequencies but not for tags or entities.
As always, questions of data, questions of data, method, and research questions are interlinked.


📝 Whatever you choose, the same questions apply:
Where did the text come from?
What was the model trained on?
How would you check it against human (e.g., your own) judgement?

### ⭐ A brief sentiment demo

We use VADER (`vaderSentiment`): a rule-based scorer with a word list (a *lexicon*) of rated words.
Its **compound** score runs from -1 (most negative) to +1 (most positive).
It was designed with modern social-media text in mind, which is not what most of our texts are.

In [ ]:
%pip install vaderSentiment

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

for sentence in [
    "Entry costs 12 pounds for adults, and children go free.",
    "The concert was lovely, and the crowd applauded with delight.",
    "A sad day today; news came that poor Tom Marlow Esq. is dead of the plague in Southwark.",
]:
    print(analyzer.polarity_scores(sentence)["compound"], sentence)

Does each score match your own reading?
1. With the version we used, the first sentence is broadly neutral (plain information),
yet it scores about +0.5 because "free" is a positive word in the lexicon.
Maybe that's correct (children getting free entry is good?), or maybe it's neutral?
We don't even know what the entry is to after all!
Either way, make up your own mind and test variants.
Try it again, removing `, and children go free.` to see how it changes.
2. Clearly positive.
3. Clearly negative, which seems right.

Sentiment scores describe the *words in a sentence*, not what the writer felt,
and there are many ways it can fail the human test.
We've seen one above.
Others include archaic or unusual words which may be missing from the lexicon.
As with entities, check a tool against your own reading on a sample *before* building anything on it.

---
## ⚠️ Common Errors

| Symptom | What to check |
|---|---|
| `OSError: [E050] Can't find model 'en_core_web_sm'` | Run the model download cell, then restart the kernel and re-run from the imports |
| `ModuleNotFoundError: No module named 'spacy'` | Run `%pip install spacy` and restart the kernel |
| `ValueError: [E088] Text of length ... exceeds maximum` | The text is very long: process it in parts, or raise `nlp.max_length` deliberately |
| `ValueError: [E1041] Expected a string ...` | You passed something that is not text to `nlp`, often `None` after a failed download |
| `URLError`, `HTTPError` or a timeout when downloading | Check your internet connection and the address. Some sites block requests with no `User-Agent`. The site may also be down or may have changed |
| `KeyError: 'extract'` (Wikisource) | The page title was probably not found: check it exactly, including capitals and punctuation |
| `UnicodeDecodeError` | Decode as UTF-8, or check which encoding the source uses |
| Licence text, the title or markup show up as entities | The header or footer was not stripped. Look at the start of your text before analysing |
| `token.lemma` prints a big number | Use `token.lemma_` (with the underscore) for text |
| Different results from someone else | Compare spaCy and model versions (printed in the setup) |

**The cells below fail on purpose.**
Read each message, work out the cause, then compare with the explanation underneath.

### ⚠️ Deliberate error 1: a model that does not exist

In [ ]:
spacy.load("en_core_web_nonexistent")

`OSError: [E050] Can't find model`: the name does not match any installed model.
The usual cause in real use is that the model was never downloaded, or that the notebook was not restarted after downloading it.

### ⚠️ Deliberate error 2: a text that is too long

In [ ]:
tiny = spacy.blank("en")
tiny.max_length = 10
tiny("This text is longer than ten characters.")

`ValueError: [E088] Text of length ... exceeds maximum`: spaCy refuses very long texts by default to protect your memory.
Here we set the limit absurdly low to trigger it. In real use, split a very long text into chapters or paragraphs.

### ⚠️ Deliberate error 3: a failed download

In [ ]:
text_from_failed_download = None
nlp(text_from_failed_download)

`ValueError: [E1041] Expected a string, Doc, or bytes as input, but got NoneType`: the variable holds `None`, not text.
In Part 5 a failed download is caught and replaced by invented text; without that step, `None` would reach `nlp`.

---
## 🟢 Information: Summary

| To ... | Write ... |
|---|---|
| Install the model | `spacy.cli.download("en_core_web_sm")` |
| Load and run the pipeline | `nlp = spacy.load("en_core_web_sm")`; `doc = nlp(text)` |
| Sentences | `doc.sents` |
| Token attributes | `token.text`, `token.lemma_`, `token.pos_`, `token.is_stop` |
| Entities and their types | `ent.text`, `ent.label_`; `spacy.explain(label)` |
| Compare two collections | sets: `a & b` (in both), `a - b` (only in `a`) |
| Evaluate a model | hand-label a sample; precision = tp / (tp + fp), recall = tp / (tp + fn) |
| Count lemmas | `Counter()` and `counts[lemma] += 1` |
| Compare texts of different length | rate per 1,000 tokens |
| Add rules | `nlp.add_pipe("entity_ruler", before="ner")` |
| Download a text | `urlopen(Request(url, headers=...))`, then clean it |

A model's output is a claim to test.
Evaluate on a sample you labelled yourself, say how big and how chosen that sample was, and record the versions you used.
A model that works well on simple modern sentences can still stumble on London place names, in any period.

**Final reflection:**
- What kinds of London names did the model handle, and which did it miss? Why?
- Was the main problem the *period* of the text, or something else?
- What would you need to do before trusting an entity count for a research claim?
- Whose decisions are built into a rule list, a gold standard, or a sentiment lexicon?

### Source and documentation

- [spaCy documentation](https://spacy.io/usage/spacy-101)
- [spaCy models](https://spacy.io/models)
- [spaCy `EntityRuler`](https://spacy.io/api/entityruler)
- [VADER sentiment (`vaderSentiment`)](https://github.com/cjhutto/vaderSentiment)

All texts in Parts 1 to 4 are invented for this course. They are not by any real author, and the people in them do not exist.
Real texts are not included in this repository: Part 5 shows how to download them, and what to check first.

End

---